# 05 · Market Structure Analysis

Analysis of concentration, specialisation, and hierarchical organisation
in the Portuguese Public Procurement market (2011-2022).

**Sections:**
1. Market concentration (HHI, Pareto share)
2. Geographic & service specialisation
3. Procedure-type breakdown
4. Legislature-level comparisons
5. Sub-market (community) profiling

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 4)})
print('Libraries loaded.')


## 1 · Market Concentration

> The Portuguese public procurement market is highly concentrated:
> **2.2 % of firms account for 80 % of total revenues** (Sturm et al., 2025).

In [ ]:
# ── Synthetic dataset (replace with real data) ────────────────────────
np.random.seed(0)
N_FIRMS = 17_397
revenues = np.random.pareto(1.2, N_FIRMS) * 1_000
revenues = np.sort(revenues)[::-1]

cum_share = np.cumsum(revenues) / revenues.sum()
pct_firms  = np.arange(1, N_FIRMS + 1) / N_FIRMS * 100

# Pareto share: how many % of firms cover 80 % of revenue
idx_80 = np.searchsorted(cum_share, 0.80)
pareto_pct = pct_firms[idx_80]
print(f'Top {pareto_pct:.1f} % of firms → 80 % of revenue  (paper: 2.2 %)')

# Herfindahl-Hirschman Index
mkt_shares = revenues / revenues.sum()
HHI = np.sum(mkt_shares ** 2)
print(f'HHI = {HHI:.4f}  (1 = monopoly, 0 = perfect competition)')


### Lorenz Curve

In [ ]:
fig, ax = plt.subplots()
ax.plot(pct_firms, cum_share * 100, label='Procurement revenue')
ax.plot([0, 100], [0, 100], 'k--', lw=0.8, label='Perfect equality')
ax.axvline(pareto_pct, color='red', ls=':', lw=1,
           label=f'80 % revenue @ top {pareto_pct:.1f} % firms')
ax.set_xlabel('Cumulative % of firms (ranked by revenue)')
ax.set_ylabel('Cumulative % of revenue')
ax.set_title('Market Concentration — Lorenz Curve')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('lorenz_curve.png', bbox_inches='tight')
plt.show()
print('Figure saved.')


## 2 · Geographic & Service Specialisation (HHI per firm)

In [ ]:
# Each firm's geographic specialisation is measured by a firm-level HHI
# over NUTS-2 regions where it has won contracts.
regions  = ['PT11','PT15','PT16','PT17','PT18','PT20','PT30']
n_firms  = 500

# Simulate region-share matrix
rng = np.random.default_rng(1)
shares = rng.dirichlet(np.ones(len(regions)) * 0.5, size=n_firms)
hhi_geo = np.sum(shares ** 2, axis=1)

print(f'Median geographic HHI : {np.median(hhi_geo):.3f}')
print(f'Mean   geographic HHI : {np.mean(hhi_geo):.3f}')

fig, ax = plt.subplots()
ax.hist(hhi_geo, bins=30, edgecolor='white')
ax.set_xlabel('Geographic HHI (per firm)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Firm-Level Geographic Specialisation')
plt.tight_layout()
plt.savefig('hhi_geo_dist.png', bbox_inches='tight')
plt.show()


## 3 · Procedure-Type Breakdown

> Between 2018-2022, **73–78 % of contracts** (by count) were awarded through
> non-competitive procedures, while competitive (open tender) procedures
> accounted for **30–38 % by value** (Sturm et al., 2025).

In [ ]:
procedure_data = {
    'Procedure'      : ['Direct Award', 'Prior Consultation', 'Open Tender'],
    'Share_by_count' : [0.65,  0.10,  0.25],
    'Share_by_value' : [0.30,  0.05,  0.65],
}
df_proc = pd.DataFrame(procedure_data)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col, title in zip(axes,
                           ['Share_by_count', 'Share_by_value'],
                           ['By number of contracts', 'By contract value']):
    ax.bar(df_proc['Procedure'], df_proc[col])
    ax.set_title(title)
    ax.set_ylabel('Share')
    ax.set_ylim(0, 1)
plt.suptitle('Procedure-Type Breakdown (2018-2022 approximation)')
plt.tight_layout()
plt.savefig('procedure_breakdown.png', bbox_inches='tight')
plt.show()


## 4 · Legislature-Level Comparison

Network statistics per legislature (from Table 1, Sturm et al., 2025).

In [ ]:
stats = pd.DataFrame({
    'Legislature'   : ['XII (2011-15)', 'XIII (2015-19)', 'XIV (2019-22)'],
    'n_nodes'       : [1987, 2213, 2214],
    'n_edges'       : [37212, 38577, 34685],
    'modularity'    : [0.71, 0.77, 0.80],
    'n_communities' : [32, 35, 37],
    'median_degree' : [23, 23, 23],
})

print(stats.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, col, lbl in zip(axes,
                         ['n_nodes', 'modularity', 'n_communities'],
                         ['# Nodes', 'Modularity', '# Communities']):
    ax.bar(stats['Legislature'], stats[col], color=['#4C72B0','#DD8452','#55A868'])
    ax.set_title(lbl)
    ax.set_xticklabels(stats['Legislature'], rotation=15, ha='right', fontsize=8)
plt.suptitle('Network Statistics by Legislature')
plt.tight_layout()
plt.savefig('legislature_stats.png', bbox_inches='tight')
plt.show()


## 5 · Sub-Market (Community) Profiling

In [ ]:
# Top-10 communities from the XII legislature network
# (labels inferred from Fig. 2 and Sect. 4 of the paper)
communities = pd.DataFrame({
    'Community' : [f'C1_{i}' for i in range(1, 11)],
    'Label'     : [
        'Construction (regional A)', 'Construction (regional B)',
        'IT Services', 'Medical Supplies (Continent)',
        'Construction (regional C)', 'Construction (regional D)',
        'Office Supplies', 'Consulting',
        'Transport', 'Medical Supplies (Islands)'
    ],
    'Size' : [220, 195, 180, 160, 140, 120, 100, 90, 80, 70],
})

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(communities['Label'][::-1], communities['Size'][::-1])
ax.set_xlabel('Approximate community size (# firms)')
ax.set_title('Top-10 Communities — XII Legislature Network')
plt.tight_layout()
plt.savefig('community_sizes.png', bbox_inches='tight')
plt.show()

print('\nCommunity profile table:')
print(communities.to_string(index=False))
